# 2.0 - Extract Embeddings

Load the Task-0 ResNet50 from Drive, check it still works, then run it once over every
clip and save the per-frame features.

```
Diving_01.pt   (16, 3, 224, 224)   9.63 MB
      |  frozen ResNet50, one time
      v
Diving_01      (16, 2048)          65.5 KB      147x smaller
```

The ResNet is frozen after notebook 1, so it gives the same answer for the same clip
every epoch. Caching it makes training about 40x faster.

We cache at 2048, before the LSTM, not at the LSTM's 256-d output. Caching 256 would
freeze the LSTM forever and it could never learn the new classes.

Two checks run here, and they answer different questions:

| Check | Question | If it fails |
|---|---|---|
| teacher accuracy on base test | did the checkpoint load correctly? | wrong path or wrong architecture |
| probe on the cache | do the frozen features carry the NEW classes? | base classes too narrow |

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "xx"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Preprocess data**

In [ ]:
# BASE
cfg.OUTPUT_ROOT = cfg.BASE_ROOT
preprocess_dataset(splits=["train", "val", "test"], target_classes=cfg.SELECTED_CLASSES)

# TASK 1
cfg.OUTPUT_ROOT = cfg.TASK1_ROOT
preprocess_dataset(splits=["train", "val", "test"], target_classes=cfg.TASK_1)

# TASK 2
cfg.OUTPUT_ROOT = cfg.TASK2_ROOT
preprocess_dataset(splits=["train", "val", "test"], target_classes=cfg.TASK_2)

In [ ]:
cfg.SELECTED_CLASSES

In [ ]:
print(cfg.SELECTED_CLASSES + cfg.TASK_1)

In [ ]:
# This notebook writes the cache, so the label space is fixed here.
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2

class_to_idx = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}

print(f"\nclass_to_idx:\n{class_to_idx}")

In [ ]:
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2"]

print(f"Base : {len(cfg.SELECTED_CLASSES)} classes  labels 0-{len(cfg.SELECTED_CLASSES)-1}")
print(f"Task1: {len(cfg.TASK_1)} classes  labels {len(cfg.SELECTED_CLASSES)}-{len(cfg.SELECTED_CLASSES)+len(cfg.TASK_1)-1}")
print(f"Task2: {len(cfg.TASK_2)} classes  labels {len(cfg.SELECTED_CLASSES)+len(cfg.TASK_1)}-{len(ALL_CLASSES_LIST)-1}")
print(f"TOTAL: {len(ALL_CLASSES_LIST)} classes")

## **Create Loaders**

Loaders over the raw `.pt` clips. They are used twice: once to test the teacher, once
to extract the features. After that nothing reads video again.

In [ ]:
CLIP_BATCH = cfg.BATCH_SIZE
PIN_MEMORY = device.type == "cuda"

# BASE
base_train_ds, base_train_loader = build_clip_loader(cfg.BASE_ROOT, cfg.SELECTED_CLASSES, class_to_idx, "train", CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)
base_val_ds,   base_val_loader   = build_clip_loader(cfg.BASE_ROOT, cfg.SELECTED_CLASSES, class_to_idx, "val",   CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)
base_test_ds,  base_test_loader  = build_clip_loader(cfg.BASE_ROOT, cfg.SELECTED_CLASSES, class_to_idx, "test",  CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)

# TASK 1
task1_train_ds, task1_train_loader = build_clip_loader(cfg.TASK1_ROOT, cfg.TASK_1, class_to_idx, "train", CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)
task1_val_ds,   task1_val_loader   = build_clip_loader(cfg.TASK1_ROOT, cfg.TASK_1, class_to_idx, "val",   CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)
task1_test_ds,  task1_test_loader  = build_clip_loader(cfg.TASK1_ROOT, cfg.TASK_1, class_to_idx, "test",  CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)

# TASK 2
task2_train_ds, task2_train_loader = build_clip_loader(cfg.TASK2_ROOT, cfg.TASK_2, class_to_idx, "train", CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)
task2_val_ds,   task2_val_loader   = build_clip_loader(cfg.TASK2_ROOT, cfg.TASK_2, class_to_idx, "val",   CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)
task2_test_ds,  task2_test_loader  = build_clip_loader(cfg.TASK2_ROOT, cfg.TASK_2, class_to_idx, "test",  CLIP_BATCH, cfg.NUM_WORKERS, PIN_MEMORY)

In [ ]:
print(f"{'split':<8}{'Base':>8}{'Task1':>8}{'Task2':>8}")
print("-" * 32)
print(f"{'train':<8}{len(base_train_ds):>8}{len(task1_train_ds):>8}{len(task2_train_ds):>8}")
print(f"{'val':<8}{len(base_val_ds):>8}{len(task1_val_ds):>8}{len(task2_val_ds):>8}")
print(f"{'test':<8}{len(base_test_ds):>8}{len(task1_test_ds):>8}{len(task2_test_ds):>8}")
print("-" * 32)
print(f"total clips: {len(base_train_ds) + len(base_val_ds) + len(base_test_ds) + len(task1_train_ds) + len(task1_val_ds) + len(task1_test_ds) + len(task2_train_ds) + len(task2_val_ds) + len(task2_test_ds)}")

## **Upload Model**

The ResNet50 + LSTM fine-tuned on the 10 base classes in notebook 1, loaded from Drive
and frozen. `remap_teacher_checkpoint` renames the old `resnet.*` keys to `backbone.*`
and drops the `num_batches_tracked` buffers.

Any missing key that is not a buffer means the checkpoint does not match the
architecture, so it raises instead of loading a half-random model.

In [ ]:
print(f"checkpoint: {cfg.RESNET50_PATH}")
print(f"exists    : {os.path.exists(cfg.RESNET50_PATH)}")

In [ ]:
teacher = ResNet50LSTMTeacher(
    hidden_size = cfg.TEACHER_HIDDEN,
    num_classes = len(cfg.SELECTED_CLASSES),
    dropout_p   = cfg.DROPOUT_P,
).to(device)

mapped = remap_teacher_checkpoint(torch.load(cfg.RESNET50_PATH, map_location=device))
missing, unexpected = teacher.load_state_dict(mapped, strict=False)

real_missing = [k for k in missing if "num_batches_tracked" not in k]
if real_missing:
    raise RuntimeError(f"Checkpoint missing keys: {real_missing[:5]}")

print(f"missing   : {len(real_missing)}")
print(f"unexpected: {len(unexpected)}")

In [ ]:
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()

model_teacher = ModelWrapper(teacher).to(device)

print("Teacher frozen")

In [ ]:
n_backbone = sum(p.numel() for p in teacher.backbone.parameters())
n_lstm     = sum(p.numel() for p in teacher.lstm.parameters())
n_fc       = sum(p.numel() for p in teacher.fc.parameters())

print(f"backbone : {n_backbone:>12,}")
print(f"lstm     : {n_lstm:>12,}")
print(f"fc       : {n_fc:>12,}")
print(f"total    : {n_backbone + n_lstm + n_fc:>12,}")
print(f"trainable: {sum(p.numel() for p in teacher.parameters() if p.requires_grad):>12,}")
print(f"\nfc: in={teacher.fc.in_features}  out={teacher.fc.out_features}")

## **Check the Model**

Before extracting anything, confirm the checkpoint actually works. This runs the whole
teacher, backbone and LSTM and classifier, on the base test set.

It should be around 98%. If it is not, the checkpoint is wrong and every number after
this point is meaningless, so stop here rather than caching features from a broken
model.

Only the 10 base classes are tested, because that is all the teacher's `fc` was trained
to output.

In [ ]:
base_acc, base_preds, base_labels = test_model(model_teacher, base_test_loader, device)

print(f"\nBase test accuracy: {base_acc:.2%}")
print("Checkpoint looks correct." if base_acc > 0.90 else "LOW - check the checkpoint path and architecture.")

In [ ]:
plot_confusion_matrix(
    base_labels, base_preds,
    num_classes  = len(cfg.SELECTED_CLASSES),
    classes_list = cfg.SELECTED_CLASSES,
    name         = "Teacher on Base test",
    save_path    = f"{cfg.RESULTS_DIR}/cm_teacher_base",
)

In [ ]:
print_detailed_metrics(
    all_labels   = base_labels,
    all_preds    = base_preds,
    num_classes  = len(cfg.SELECTED_CLASSES),
    classes_list = cfg.SELECTED_CLASSES,
)

## **Extract Embeddings**

Run the frozen backbone over every clip and keep the per-frame `(16, 2048)` output.

Splits are cached separately and never mixed: train fits, val selects, test reports.
Stored as fp16 to halve the file, cast back to fp32 at load time because the LSTM
needs fp32.

In [ ]:
STORE_DTYPE = getattr(torch, cfg.STORE_DTYPE)

FEATS = {}

In [ ]:
# BASE
base_train_f, base_train_y = extract_frame_features(teacher.backbone, base_train_loader, device, cfg.BACKBONE_DIM, STORE_DTYPE)
base_val_f,   base_val_y   = extract_frame_features(teacher.backbone, base_val_loader,   device, cfg.BACKBONE_DIM, STORE_DTYPE)
base_test_f,  base_test_y  = extract_frame_features(teacher.backbone, base_test_loader,  device, cfg.BACKBONE_DIM, STORE_DTYPE)

FEATS["t0_train"] = (base_train_f, base_train_y)
FEATS["t0_val"]   = (base_val_f,   base_val_y)
FEATS["t0_test"]  = (base_test_f,  base_test_y)

print(f"t0_train {tuple(base_train_f.shape)}   t0_val {tuple(base_val_f.shape)}   t0_test {tuple(base_test_f.shape)}")

In [ ]:
# TASK 1
task1_train_f, task1_train_y = extract_frame_features(teacher.backbone, task1_train_loader, device, cfg.BACKBONE_DIM, STORE_DTYPE)
task1_val_f,   task1_val_y   = extract_frame_features(teacher.backbone, task1_val_loader,   device, cfg.BACKBONE_DIM, STORE_DTYPE)
task1_test_f,  task1_test_y  = extract_frame_features(teacher.backbone, task1_test_loader,  device, cfg.BACKBONE_DIM, STORE_DTYPE)

FEATS["t1_train"] = (task1_train_f, task1_train_y)
FEATS["t1_val"]   = (task1_val_f,   task1_val_y)
FEATS["t1_test"]  = (task1_test_f,  task1_test_y)

print(f"t1_train {tuple(task1_train_f.shape)}   t1_val {tuple(task1_val_f.shape)}   t1_test {tuple(task1_test_f.shape)}")

In [ ]:
# TASK 2
task2_train_f, task2_train_y = extract_frame_features(teacher.backbone, task2_train_loader, device, cfg.BACKBONE_DIM, STORE_DTYPE)
task2_val_f,   task2_val_y   = extract_frame_features(teacher.backbone, task2_val_loader,   device, cfg.BACKBONE_DIM, STORE_DTYPE)
task2_test_f,  task2_test_y  = extract_frame_features(teacher.backbone, task2_test_loader,  device, cfg.BACKBONE_DIM, STORE_DTYPE)

FEATS["t2_train"] = (task2_train_f, task2_train_y)
FEATS["t2_val"]   = (task2_val_f,   task2_val_y)
FEATS["t2_test"]  = (task2_test_f,  task2_test_y)

print(f"t2_train {tuple(task2_train_f.shape)}   t2_val {tuple(task2_val_f.shape)}   t2_test {tuple(task2_test_f.shape)}")

The Task-0 LSTM weights are saved next to the features, so every later notebook starts
its LSTM from the same place.

In [ ]:
save_feature_cache(
    cfg.FEAT_CACHE, FEATS, teacher.lstm.state_dict(),
    cfg.BACKBONE_DIM, cfg.CLIP_LEN, ALL_CLASSES_LIST, TASK_NAMES,
)

In [ ]:
del teacher
del model_teacher
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Teacher released")

## **Verify Cache**

Load it back from Drive, exactly the way notebooks 4.1 to 4.6 will, and check the
shapes and that no task's labels leaked into another task's tensors.

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

print(f"\ncached label space: {META['global_classes']}")
print(f"task names        : {META['task_names']}")

In [ ]:
print_cache_summary(CACHE, TASK_NAMES)

In [ ]:
verify_cache(CACHE, TASK_CLASSES, class_to_idx, cfg.BACKBONE_DIM, cfg.CLIP_LEN)

## **Embedding Quality**

The ResNet was fine-tuned on 10 sports classes and then frozen. Do its features still
carry the new classes, makeup and boxing, or did the fine-tuning throw that away?

Each clip is mean-pooled from `(16, 2048)` to `(2048,)` and a simple classifier is
asked to separate the classes. 1-NN is pure geometry with nothing learned; the linear
probe measures how much linearly separable information is there.

The **joint linear probe is the ceiling**. If it reads 90%, then no continual-learning
method in notebook 4 can beat 90%, and every result there is read against this number.

In [ ]:
report = probe_report(CACHE, TASK_NAMES, k=1)

Silhouette score, in [-1, 1], says how tight and well separated the class clusters are.
Near 0 means clusters overlap heavily.

In [ ]:
task_ids = [0, 1, 2]

sil_base  = separability(CACHE, [0])
sil_task1 = separability(CACHE, [1])
sil_task2 = separability(CACHE, [2])
sil_joint = separability(CACHE, task_ids)

print(f"{'Group':<10}{'silhouette':>12}")
print("-" * 22)
print(f"{'Base':<10}{sil_base:>12.3f}")
print(f"{'Task1':<10}{sil_task1:>12.3f}")
print(f"{'Task2':<10}{sil_task2:>12.3f}")
print(f"{'JOINT':<10}{sil_joint:>12.3f}")

Classes whose centroids sit close together are the ones that will compete during
continual learning. A close pair that crosses tasks predicts exactly where the
confusion matrices in notebook 4 will go wrong, before any training happens.

In [ ]:
cents        = class_centroids(CACHE, task_ids, "train")
labels, dist = centroid_distances(cents)

print_closest_pairs(cents, ALL_CLASSES_LIST, TASK_CLASSES, TASK_NAMES, top=10)

## **Latent Space**

Four views of the same cache: the probe scores, the features coloured by task, the same
features coloured by class, and the centroid distance heatmap.

In [ ]:
task_of = {}
for t, group in enumerate(TASK_CLASSES):
    for c in group:
        task_of[class_to_idx[c]] = t

xy, lab = project_2d(CACHE, task_ids, split="train", method="tsne", seed=cfg.SEED)

print(f"projected {len(xy)} clips")

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs  = fig.add_gridspec(2, 2, hspace=0.25, wspace=0.2)

plot_probe_bars(report, ax=fig.add_subplot(gs[0, 0]))
plot_latent_space(xy, lab, ALL_CLASSES_LIST, task_of=task_of,
                  title="Frozen features by task", ax=fig.add_subplot(gs[0, 1]))
plot_latent_space(xy, lab, ALL_CLASSES_LIST,
                  title="Frozen features by class", ax=fig.add_subplot(gs[1, 0]))
plot_centroid_heatmap(labels, dist, ALL_CLASSES_LIST, ax=fig.add_subplot(gs[1, 1]))

plt.savefig(f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## **Save**

The cache itself is already on Drive. This saves the diagnostics next to it so notebook
4 can quote the ceiling without recomputing it.

In [ ]:
diag = {
    "stage":            "2_embedding_quality",
    "classes":          ALL_CLASSES_LIST,
    "task_names":       TASK_NAMES,
    "teacher_base_acc": float(base_acc),
    "probe":            report,
    "silhouette":       {"Base": sil_base, "Task1": sil_task1, "Task2": sil_task2},
    "silhouette_joint": sil_joint,
    "ceiling":          report["JOINT"]["linear"],
    "n_clips":          int(sum(f.shape[0] for f, _ in CACHE.values())),
}

os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2, default=float)

print(f"Saved -> {cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json")

In [ ]:
print(f"cache            : {cfg.FEAT_CACHE}")
print(f"teacher base acc : {base_acc:.2%}")
print(f"JOINT 1-NN       : {report['JOINT']['knn']:.2%}")
print(f"JOINT linear     : {report['JOINT']['linear']:.2%}   <- ceiling for notebook 4")
print(f"chance           : {report['JOINT']['chance']:.2%}")

---

Next: `4_continual_learning/`. Run 4.1 first, it trains and saves the shared
`head_base.pt`, then 4.2 to 4.6 in any order.